In [39]:
import json
import os

import nltk
import preprocessor as p
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

[nltk_data] Downloading package punkt_tab to /home/ddore/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/ddore/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /home/ddore/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /home/ddore/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [40]:
def clean_text(content):
    """Cleans text content by removing unwanted characters and formatting issues."""
    content = str(content)
    content = re.sub(r'<[^>]+>', '', content)
    content = re.sub(r'[\t\n\r\f\v]+', ' ', content)
    content = re.sub(r' {2,}', ' ', content)
    try:
        content = content.split(":", 1)[1].strip()
    except:
        pass
    content = p.clean(content)
    return content.strip()

In [41]:
path = "/workspace/ddore/03-fallacy-rag/FallacyRAG/data/"
files = [path + f for f in os.listdir(path) if f.endswith(".txt")]
files[0]

'/workspace/ddore/03-fallacy-rag/FallacyRAG/data/5_1976.txt'

In [42]:
docs = []
for f in files:
    with open(f, 'r') as file_object:
        sentences = file_object.read().split("\n")
        sentence = ""
        for s in sentences:
            sentence += clean_text(s)
        docs.append(clean_text(sentence))
len(docs)

45

## NER

In [43]:
entities = {}
for idx, sentence in enumerate(docs):
    tokens = word_tokenize(sentence)
    pos_tags = pos_tag(tokens)
    named_entities = ne_chunk(pos_tags)

    for chunk in named_entities:
        if isinstance(chunk, Tree):
            entity_label = chunk.label()
            entity_text = " ".join(c[0] for c in chunk)

            entities.setdefault(entity_label, set()).add(entity_text)

serializable_entities = {label: list(entities[label]) for label in entities}
print(json.dumps(serializable_entities, indent=4))

{
    "GPE": [
        "Torah",
        "Western",
        "Sandinista",
        "Iraq.Wrong.And",
        "Obama",
        "Bean",
        "Abkhazia",
        "Chicago",
        "Beaumont",
        "Leninism",
        "Bassad",
        "Arctic",
        "Budapest",
        "America.The",
        "Morton",
        "Americas",
        "Virginia",
        "Iraq.That",
        "South",
        "Long",
        "California",
        "Europeans",
        "Maine",
        "Airbus",
        "Rumania",
        "Drugs",
        "Washington",
        "Drop",
        "New Mexico",
        "Reed",
        "Im",
        "Syrian",
        "Appalachian",
        "Qataris",
        "Baker",
        "Middle East",
        "Poles",
        "Child",
        "Afghanistan.Afghanistan",
        "Hoover",
        "Briefly",
        "Mondale",
        "Independents",
        "Illinois",
        "Iranian",
        "Latin America",
        "Alden",
        "July",
        "Korea",
        "South Koreans",
      

## Topic Extraction

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [45]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
model.to(device)
model.eval()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [46]:
import re
import torch

SYSTEM_MSG = "You are a political debate topic classifier. Follow the user's format exactly."

USER_TEMPLATE = """Task:
Identify the main topics discussed in this debate.

Return ONLY the topic names separated by commas in this exact format:
<TOPICS: topic1, topic2, ...>

Do not output anything else.

Debate:
{chunk}
"""

In [47]:
def chunk_by_tokens(text: str, tokenizer, max_input_tokens: int):
    ids = tokenizer.encode(text, add_special_tokens=False)
    for i in range(0, len(ids), max_input_tokens):
        chunk_ids = ids[i:i + max_input_tokens]
        yield tokenizer.decode(chunk_ids)


def extract_topics(raw: str):
    TOPIC_RE = re.compile(r"TOPICS\s*:\s*(.+)", re.IGNORECASE)
    m = TOPIC_RE.search(raw)
    if not m:
        return []
    items = [t.replace("-", " ").replace("_", " ").strip() for t in m.group(1).split(",")]
    # normalize + drop empties
    return [t for t in (i.lower().replace(" ", "_") for i in items) if t]

In [48]:
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 128

all_topics = set()
for debate in docs:
    for chunk in chunk_by_tokens(debate, tokenizer, MAX_INPUT_TOKENS):
        user_prompt = USER_TEMPLATE.format(chunk=chunk)

        messages = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": user_prompt},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.0,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

        topics = extract_topics(raw)
        all_topics.update(topics)

print("\nFINAL TOPICS:", sorted(all_topics))


FINAL TOPICS: ['9/11', '9/11_terrorism', 'aarp', 'abortion', 'abortion_legal_status', 'abortion_rights', 'abraham_lincoln', 'accountability', 'accounting', 'achievement_gap', 'acid_rain_legislation', 'acorn', 'acorn_controversy', 'administration_policy', 'administrative_costs', 'administrative_performance', 'adoption', 'adversary_relations', 'afdc', 'affirmative_action', 'affordable_care_act', 'afghan_forces', 'afghanistan', 'afghanistan_policy', 'afghanistan_security', 'afghanistan_stability', 'afghanistan_timeline', 'afghanistan_war', 'africa', 'african_policy', 'age_of_leader', 'agingpopulation', 'agricultural_exports', 'agricultural_policy', 'agriculture', 'agriculture_subsidies', 'aid', 'aid_to_pakistan', 'aids', 'aids_epidemic', 'aids_research', 'aircraft_programs', 'airline_industry', 'airport_security', 'al_qaeda', 'al_qaida', 'al_qaida_connection', 'aleppo', 'alliance_building', 'alliance_for_progress', 'alliances', 'alliances_and_international_relations', 'allies', 'alternat

In [49]:
entities['topics'] = all_topics

In [50]:
import json

with open('topic_results.json', 'w') as f:
    json.dump(
        entities,
        f,
        default=lambda o: sorted(o) if isinstance(o, set) else str(o),
        ensure_ascii=False,
        indent=2
    )